<a href="https://colab.research.google.com/github/Mr-Zainulabadin/python_programing/blob/main/Lab_11_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Sentiment Analysis (Text Classification)**
*   **Text Cleaning**
*   **Text Preprocessing**
*   **Feature Engineering**
*   **ML Model**

# **Importing Preprocessing Libraries**

In [4]:
!pip install contractions

import pandas as pd
import string

import re
import contractions
import nltk
#from nltk.tokenize import word_tokenize
#from nltk.corpus import stopwords
#from nltk.stem import WordNetLemmatizer


#nltk.download('wordnet')
#nltk.download('stopwords')
#nltk.download('punkt_tab')


#stopwords.words('english')

# **Reading Data**

In [12]:
import pandas as pd

temp_df = pd.read_csv(
    '/content/imdb.csv',
    engine='python',
    on_bad_lines='skip'
)

df = temp_df.iloc[:50000]

In [13]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# **Text Cleaning & Preprocessing**

In [14]:
def remove_html_tags(text):
    return re.sub(r'<.*?>', '', text)

def remove_url(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def remove_contractions(text):
  expanded_text = contractions.fix(text)
  return expanded_text

def remove_punc(text):
    unwanted = set(string.punctuation + string.digits)
    return ''.join(char for char in str(text) if char not in unwanted)

In [17]:
df['review'] = df['review'].str.lower()

df['review'] = df['review'].apply(remove_html_tags)

df['review'] = df['review'].apply(remove_url)

#df['review'] = df['review'].apply(remove_contractions)

#df['review'] = df['review'].apply(remove_punc)

#df['review'] = df['review'].apply(word_tokenize)

#df['review'] = df['review'].apply(remove_stopwords)

#df['review'] = df['review'].apply(lemmatize_words)

In [18]:
df['review']

,review
0,one of the other reviewers has mentioned that ...
1,a wonderful little production. the filming tec...
2,i thought this was a wonderful way to spend ti...
3,basically there's a family where a little boy ...
4,"petter mattei's ""love in the time of money"" is..."
...,...
22145,"rich, alcoholic robert stack falls in love wit..."
22146,"in an era of such awful cartoons, i am rather ..."
22147,"well, at least this was the last sequel that i..."
22148,"oh my god... where to begin? ""chupacabra terro..."


# **Feature Engineering**

**Target Column Encoding**

In [19]:
from sklearn.preprocessing import LabelEncoder

#X = df.drop('sentiment', axis=1)
X = df['review']
Y = df['sentiment']

print(X)
print(Y)

encoder = LabelEncoder()
Y = encoder.fit_transform(Y)

print(Y)

0        one of the other reviewers has mentioned that ...
1        a wonderful little production. the filming tec...
2        i thought this was a wonderful way to spend ti...
3        basically there's a family where a little boy ...
4        petter mattei's "love in the time of money" is...
                               ...                        
22145    rich, alcoholic robert stack falls in love wit...
22146    in an era of such awful cartoons, i am rather ...
22147    well, at least this was the last sequel that i...
22148    oh my god... where to begin? "chupacabra terro...
22149    this movie is extremely boring, it tells a sto...
Name: review, Length: 22150, dtype: object
0        positive
1        positive
2        positive
3        negative
4        positive
           ...   
22145    positive
22146    positive
22147    negative
22148    negative
22149    negative
Name: sentiment, Length: 22150, dtype: object
[1 1 1 ... 0 0 0]


**Bag of Words**

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,confusion_matrix

X_train,X_test,y_train,y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

print(X_train.shape)
#print(X_train.head)

#print(X_train)
#print(X_test)

vectorizer = CountVectorizer()

X_train_bow = vectorizer.fit_transform(X_train)

X_test_bow = vectorizer.transform(X_test)

# Output the shapes of the resulting Bag of Words matrices
print(f"Shape of X_train_bow: {X_train_bow.shape}")
print(f"Shape of X_test_bow: {X_test_bow.shape}")

# Applying Random Forest Classifier
rf = RandomForestClassifier()

rf.fit(X_train_bow,y_train)
y_pred = rf.predict(X_test_bow)
#accuracy_score(y_test,y_pred)

print (accuracy_score(y_test,y_pred))
print (confusion_matrix(y_test,y_pred))

(17720,)
Shape of X_train_bow: (17720, 67283)
Shape of X_test_bow: (4430, 67283)
0.8455981941309255
[[1885  342]
 [ 342 1861]]


**n-gram (2-gram)**

In [21]:
cv = CountVectorizer(ngram_range=(2,2))

X_train_n_gram = cv.fit_transform(X_train)
X_test_n_gram = cv.transform(X_test)

# Output the shapes of the resulting Bag of Words matrices
print(f"Shape of X_train_bow: {X_train_n_gram.shape}")
print(f"Shape of X_test_bow: {X_test_n_gram.shape}")

rf = RandomForestClassifier()

rf.fit(X_train_n_gram,y_train)
y_pred = rf.predict(X_test_n_gram)

print (accuracy_score(y_test,y_pred))
print (confusion_matrix(y_test,y_pred))

Shape of X_train_bow: (17720, 1118669)
Shape of X_test_bow: (4430, 1118669)
0.8261851015801355
[[1840  387]
 [ 383 1820]]


**TF/IDF**

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Output the shapes of the resulting Bag of Words matrices
print(f"Shape of X_train_bow: {X_train_tfidf.shape}")
print(f"Shape of X_test_bow: {X_test_tfidf.shape}")

rf = RandomForestClassifier()

rf.fit(X_train_tfidf,y_train)
y_pred = rf.predict(X_test_tfidf)

print (accuracy_score(y_test,y_pred))
print (confusion_matrix(y_test,y_pred))

Shape of X_train_bow: (17720, 67283)
Shape of X_test_bow: (4430, 67283)
0.8329571106094809
[[1896  331]
 [ 409 1794]]


# **Tasks:**
*   **Add a Python Function to remove Stop Words from the IMDB reviews data.**
*   **After Stopword Removal, add a Python Function to perform Lemmitization over IMDB Reviews data.**

**After applying the above mentioned data preprocessing steps, again run this code and analyse the performance of the ML models for text classification of IMDB Reviews.**

**Apply the ML classifier on the following dataset. https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset**




# IMPORT LIBRARIES

In [48]:
!pip install contractions


import pandas as pd
import numpy as np
import re
import string
import contractions
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [49]:
# DOWNLOAD NLTK DATA


nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# =========================================================
# LOAD DATASET
# =========================================================

temp_df = pd.read_csv(
    '/content/imdb.csv',
    engine='python',
    on_bad_lines='skip'
)

# Use first 50000 rows
df = temp_df.iloc[:50000]

print(df.head())


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [52]:
# TEXT CLEANING FUNCTIONS


# Remove HTML Tags
def remove_html_tags(text):

    pattern = re.compile('<.*?>')

    return pattern.sub('', str(text))

# Remove URLs
def remove_url(text):

    pattern = re.compile(r'https?://\S+|www\.\S+')

    return pattern.sub('', str(text))

# Remove Contractions
def remove_contractions(text):

    return contractions.fix(str(text))

# Remove Punctuation and Numbers
def remove_punc(text):

    unwanted = set(string.punctuation + string.digits)

    return ''.join(
        char for char in str(text)
        if char not in unwanted
    )

# =========================================================
# STOPWORD REMOVAL FUNCTION
# =========================================================

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):

    words = word_tokenize(text)

    filtered_words = []

    for word in words:

        if word not in stop_words:

            filtered_words.append(word)

    return " ".join(filtered_words)

# LEMMATIZATION FUNCTION


lemmatizer = WordNetLemmatizer()

def lemmatize_words(text):

    words = word_tokenize(text)

    lemmatized_words = []

    for word in words:

        lemmatized_words.append(
            lemmatizer.lemmatize(word)
        )

    return " ".join(lemmatized_words)

# APPLY PREPROCESSING

In [53]:


# Convert into lowercase
df['review'] = df['review'].str.lower()

# Remove HTML Tags
df['review'] = df['review'].apply(remove_html_tags)

# Remove URLs
df['review'] = df['review'].apply(remove_url)

# Remove Contractions
df['review'] = df['review'].apply(remove_contractions)

# Remove Punctuation
df['review'] = df['review'].apply(remove_punc)

# Remove Stopwords
df['review'] = df['review'].apply(remove_stopwords)

# Lemmatization
df['review'] = df['review'].apply(lemmatize_words)

print("\nPreprocessed Reviews:\n")
print(df['review'].head())



Preprocessed Reviews:

0    one reviewer mentioned watching oz episode hoo...
1    wonderful little production filming technique ...
2    thought wonderful way spend time hot summer we...
3    basically family little boy jake think zombie ...
4    petter matteis love time money visually stunni...
Name: review, dtype: object


# LABEL ENCODING

In [55]:


X = df['review']
Y = df['sentiment']

encoder = LabelEncoder()

Y = encoder.fit_transform(Y)

print("\nEncoded Labels:")
print(Y[:10])

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)


Encoded Labels:
[1 1 1 0 1 1 1 0 0 1]


# BAG OF WORDS

In [57]:



bow = CountVectorizer()

X_train_bow = bow.fit_transform(X_train)

X_test_bow = bow.transform(X_test)

print("Train Shape:", X_train_bow.shape)
print("Test Shape :", X_test_bow.shape)

Train Shape: (17720, 104721)
Test Shape : (4430, 104721)


# ---------------- RANDOM FOREST ----------------

In [58]:


rf_bow = RandomForestClassifier()

rf_bow.fit(X_train_bow, y_train)

y_pred_rf_bow = rf_bow.predict(X_test_bow)

evaluate_model(
    "Random Forest + BOW",
    y_test,
    y_pred_rf_bow
)


Random Forest + BOW
Accuracy : 0.8494356659142213
Precision: 0.8548983364140481
Recall   : 0.8397639582387654
F1 Score : 0.8472635676665904

Confusion Matrix:
[[1913  314]
 [ 353 1850]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.86      0.85      2227
           1       0.85      0.84      0.85      2203

    accuracy                           0.85      4430
   macro avg       0.85      0.85      0.85      4430
weighted avg       0.85      0.85      0.85      4430



# ---------------- KNN ----------------

In [59]:

knn_bow = KNeighborsClassifier(n_neighbors=5)

knn_bow.fit(X_train_bow, y_train)

y_pred_knn_bow = knn_bow.predict(X_test_bow)

evaluate_model(
    "KNN + BOW",
    y_test,
    y_pred_knn_bow
)


KNN + BOW
Accuracy : 0.6160270880361174
Precision: 0.6274111675126903
Recall   : 0.5610531093962778
F1 Score : 0.5923795830337887

Confusion Matrix:
[[1493  734]
 [ 967 1236]]

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.67      0.64      2227
           1       0.63      0.56      0.59      2203

    accuracy                           0.62      4430
   macro avg       0.62      0.62      0.61      4430
weighted avg       0.62      0.62      0.61      4430



# NGRAM (BIGRAM)

In [1]:


ngram = CountVectorizer(ngram_range=(2,2))

X_train_ngram = ngram.fit_transform(X_train)

X_test_ngram = ngram.transform(X_test)

print("Train Shape:", X_train_ngram.shape)
print("Test Shape :", X_test_ngram.shape)



NameError: name 'CountVectorizer' is not defined

In [ ]:
# ---------------- RANDOM FOREST ----------------

rf_ngram = RandomForestClassifier()

rf_ngram.fit(X_train_ngram, y_train)

y_pred_rf_ngram = rf_ngram.predict(X_test_ngram)

evaluate_model(
    "Random Forest + NGRAM",
    y_test,
    y_pred_rf_ngram
)

In [56]:


# ---------------- KNN ----------------

knn_ngram = KNeighborsClassifier(n_neighbors=5)

knn_ngram.fit(X_train_ngram, y_train)

y_pred_knn_ngram = knn_ngram.predict(X_test_ngram)

evaluate_model(
    "KNN + NGRAM",
    y_test,
    y_pred_knn_ngram
)

# TF-IDF


print("\n================================================")
print("TF-IDF")


tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

print("Train Shape:", X_train_tfidf.shape)
print("Test Shape :", X_test_tfidf.shape)





BAG OF WORDS
Train Shape: (17720, 104721)
Test Shape : (4430, 104721)


KeyboardInterrupt: 

In [ ]:
# ---------------- RANDOM FOREST ----------------

rf_tfidf = RandomForestClassifier()

rf_tfidf.fit(X_train_tfidf, y_train)

y_pred_rf_tfidf = rf_tfidf.predict(X_test_tfidf)

evaluate_model(
    "Random Forest + TFIDF",
    y_test,
    y_pred_rf_tfidf
)

# ---------------- KNN ----------------

knn_tfidf = KNeighborsClassifier(n_neighbors=5)

knn_tfidf.fit(X_train_tfidf, y_train)

y_pred_knn_tfidf = knn_tfidf.predict(X_test_tfidf)

evaluate_model(
    "KNN + TFIDF",
    y_test,
    y_pred_knn_tfidf
)



# FINAL PERFORMANCE SUMMARY

In [ ]:



models = [
    "RF + BOW",
    "KNN + BOW",
    "RF + NGRAM",
    "KNN + NGRAM",
    "RF + TFIDF",
    "KNN + TFIDF"
]

accuracies = [
    accuracy_score(y_test, y_pred_rf_bow),
    accuracy_score(y_test, y_pred_knn_bow),
    accuracy_score(y_test, y_pred_rf_ngram),
    accuracy_score(y_test, y_pred_knn_ngram),
    accuracy_score(y_test, y_pred_rf_tfidf),
    accuracy_score(y_test, y_pred_knn_tfidf)
]

f1_scores = [
    f1_score(y_test, y_pred_rf_bow),
    f1_score(y_test, y_pred_knn_bow),
    f1_score(y_test, y_pred_rf_ngram),
    f1_score(y_test, y_pred_knn_ngram),
    f1_score(y_test, y_pred_rf_tfidf),
    f1_score(y_test, y_pred_knn_tfidf)
]

for i in range(len(models)):

    print(models[i])

    print("Accuracy :", accuracies[i])

    print("F1 Score :", f1_scores[i])


In [ ]:


# =========================================================
# FUNCTION FOR MODEL EVALUATION
# =========================================================

def evaluate_model(model_name, y_test, y_pred):

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(y_test, y_pred)

    recall = recall_score(y_test, y_pred)

    f1 = f1_score(y_test, y_pred)

    print("\n======================================")
    print(model_name)
    print("======================================")

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


In [23]:
import pandas as pd
import numpy as np
import string
import re
import contractions
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [28]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [29]:
df = pd.read_csv('/content/spam.csv', encoding='latin-1')

In [30]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [31]:
df = df[['v1', 'v2']]

# Rename columns
df.columns = ['label', 'message']

print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [32]:
# TEXT CLEANING FUNCTIONS


# Remove HTML Tags
def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub('', str(text))

# Remove URLs
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub('', str(text))

# Remove Contractions
def remove_contractions(text):
    return contractions.fix(str(text))

# Remove Punctuation and Digits
def remove_punc(text):
    unwanted = set(string.punctuation + string.digits)

    return ''.join(
        char for char in str(text)
        if char not in unwanted
    )

# STOPWORD REMOVAL


stop_words = set(stopwords.words('english'))

def remove_stopwords(text):

    words = word_tokenize(text)

    filtered_words = []

    for word in words:
        if word not in stop_words:
            filtered_words.append(word)

    return " ".join(filtered_words)


In [33]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [37]:
import nltk
nltk.download('punkt_tab')
# LEMMATIZATION


lemmatizer = WordNetLemmatizer()

def lemmatize_words(text):

    words = word_tokenize(text)

    lemmatized_words = []

    for word in words:
        lemmatized_words.append(
            lemmatizer.lemmatize(word)
        )

    return " ".join(lemmatized_words)

# APPLY PREPROCESSING


# Convert into lowercase
df['message'] = df['message'].str.lower()

# Remove HTML Tags
df['message'] = df['message'].apply(remove_html_tags)

# Remove URLs
df['message'] = df['message'].apply(remove_url)

# Remove Contractions
df['message'] = df['message'].apply(remove_contractions)

# Remove Punctuation
df['message'] = df['message'].apply(remove_punc)

# Remove Stopwords
df['message'] = df['message'].apply(remove_stopwords)

# Lemmatization
df['message'] = df['message'].apply(lemmatize_words)

print("\nPreprocessed Messages:\n")
print(df['message'].head())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.



Preprocessed Messages:

0    go jurong point crazy available bugis n great ...
1                                ok lar joking wif oni
2    free entry wkly comp win fa cup final tkts st ...
3                      dun say early hor c already say
4                  nah think go usf life around though
Name: message, dtype: object


In [38]:

# LABEL ENCODING
# ham = 0
# spam = 1


encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['label'])

X = df['message']
y = df['label']

print("\nEncoded Labels:")
print(y.head())


Encoded Labels:
0    0
1    0
2    1
3    0
4    0
Name: label, dtype: int64


In [39]:

# TRAIN TEST SPLIT


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# BAG OF WORDS
print("BAG OF WORDS")

bow = CountVectorizer()

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

print("Shape of Train Data:", X_train_bow.shape)
print("Shape of Test Data:", X_test_bow.shape)


BAG OF WORDS
Shape of Train Data: (4457, 6861)
Shape of Test Data: (1115, 6861)


# RANDOM FOREST ON BOW

In [40]:



rf_bow = RandomForestClassifier()

rf_bow.fit(X_train_bow, y_train)

y_pred_rf_bow = rf_bow.predict(X_test_bow)

print("\nRandom Forest Accuracy (BOW):")
print(accuracy_score(y_test, y_pred_rf_bow))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_bow))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_bow))


Random Forest Accuracy (BOW):
0.9739910313901345

Confusion Matrix:
[[965   0]
 [ 29 121]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       965
           1       1.00      0.81      0.89       150

    accuracy                           0.97      1115
   macro avg       0.99      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115



# KNN ON BOW

In [41]:



knn_bow = KNeighborsClassifier(n_neighbors=5)

knn_bow.fit(X_train_bow, y_train)

y_pred_knn_bow = knn_bow.predict(X_test_bow)

print("\nKNN Accuracy (BOW):")
print(accuracy_score(y_test, y_pred_knn_bow))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn_bow))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn_bow))


KNN Accuracy (BOW):
0.9210762331838565

Confusion Matrix:
[[965   0]
 [ 88  62]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96       965
           1       1.00      0.41      0.58       150

    accuracy                           0.92      1115
   macro avg       0.96      0.71      0.77      1115
weighted avg       0.93      0.92      0.91      1115



# N-GRAM (BIGRAM)

In [43]:


print("\n================================================")
print("N-GRAM (2-GRAM)")
print("================================================")

ngram = CountVectorizer(ngram_range=(2,2))

X_train_ngram = ngram.fit_transform(X_train)
X_test_ngram = ngram.transform(X_test)

print("Shape of Train Data:", X_train_ngram.shape)
print("Shape of Test Data:", X_test_ngram.shape)

# =========================================================
# RANDOM FOREST ON NGRAM
# =========================================================

rf_ngram = RandomForestClassifier()

rf_ngram.fit(X_train_ngram, y_train)

y_pred_rf_ngram = rf_ngram.predict(X_test_ngram)

print("\nRandom Forest Accuracy (NGRAM):")
print(accuracy_score(y_test, y_pred_rf_ngram))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_ngram))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_ngram))


N-GRAM (2-GRAM)
Shape of Train Data: (4457, 23989)
Shape of Test Data: (1115, 23989)

Random Forest Accuracy (NGRAM):
0.957847533632287

Confusion Matrix:
[[964   1]
 [ 46 104]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98       965
           1       0.99      0.69      0.82       150

    accuracy                           0.96      1115
   macro avg       0.97      0.85      0.90      1115
weighted avg       0.96      0.96      0.95      1115



In [44]:
# KNN ON NGRAM


knn_ngram = KNeighborsClassifier(n_neighbors=5)

knn_ngram.fit(X_train_ngram, y_train)

y_pred_knn_ngram = knn_ngram.predict(X_test_ngram)

print("\nKNN Accuracy (NGRAM):")
print(accuracy_score(y_test, y_pred_knn_ngram))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn_ngram))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn_ngram))


KNN Accuracy (NGRAM):
0.9013452914798207

Confusion Matrix:
[[965   0]
 [110  40]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95       965
           1       1.00      0.27      0.42       150

    accuracy                           0.90      1115
   macro avg       0.95      0.63      0.68      1115
weighted avg       0.91      0.90      0.88      1115



# TF-IDF

In [46]:


print("TF-IDF")


tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Shape of Train Data:", X_train_tfidf.shape)
print("Shape of Test Data:", X_test_tfidf.shape)

# RANDOM FOREST ON TF-IDF


rf_tfidf = RandomForestClassifier()

rf_tfidf.fit(X_train_tfidf, y_train)

y_pred_rf_tfidf = rf_tfidf.predict(X_test_tfidf)

print("\nRandom Forest Accuracy (TF-IDF):")
print(accuracy_score(y_test, y_pred_rf_tfidf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_tfidf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_tfidf))

# KNN ON TF-IDF


knn_tfidf = KNeighborsClassifier(n_neighbors=5)

knn_tfidf.fit(X_train_tfidf, y_train)

y_pred_knn_tfidf = knn_tfidf.predict(X_test_tfidf)

print("\nKNN Accuracy (TF-IDF):")
print(accuracy_score(y_test, y_pred_knn_tfidf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn_tfidf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn_tfidf))



TF-IDF
Shape of Train Data: (4457, 6861)
Shape of Test Data: (1115, 6861)

Random Forest Accuracy (TF-IDF):
0.9730941704035875

Confusion Matrix:
[[965   0]
 [ 30 120]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       965
           1       1.00      0.80      0.89       150

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115


KNN Accuracy (TF-IDF):
0.9174887892376682

Confusion Matrix:
[[965   0]
 [ 92  58]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      1.00      0.95       965
           1       1.00      0.39      0.56       150

    accuracy                           0.92      1115
   macro avg       0.96      0.69      0.76      1115
weighted avg       0.92      0.92      0.90      1115



# FINAL COMPARISON

In [47]:

print("Random Forest + BOW Accuracy:",
      accuracy_score(y_test, y_pred_rf_bow))

print("KNN + BOW Accuracy:",
      accuracy_score(y_test, y_pred_knn_bow))

print("Random Forest + NGRAM Accuracy:",
      accuracy_score(y_test, y_pred_rf_ngram))

print("KNN + NGRAM Accuracy:",
      accuracy_score(y_test, y_pred_knn_ngram))

print("Random Forest + TFIDF Accuracy:",
      accuracy_score(y_test, y_pred_rf_tfidf))

print("KNN + TFIDF Accuracy:",
      accuracy_score(y_test, y_pred_knn_tfidf))

Random Forest + BOW Accuracy: 0.9739910313901345
KNN + BOW Accuracy: 0.9210762331838565
Random Forest + NGRAM Accuracy: 0.957847533632287
KNN + NGRAM Accuracy: 0.9013452914798207
Random Forest + TFIDF Accuracy: 0.9730941704035875
KNN + TFIDF Accuracy: 0.9174887892376682
